# Computational Toxicity Prediction with Tox21 and ToxCast

Toxicity testing has traditionally relied on animal experiments, but large-scale in-vitro screening programs like [Tox21](https://tripod.nih.gov/tox21/challenge/) and [ToxCast](https://www.epa.gov/comptox-tools/toxicity-forecasting-toxcast) have generated thousands of high-throughput assay measurements that can be used to train computational models instead. These datasets are part of a broader shift toward **New Approach Methodologies (NAMs)** -- computational and in-vitro techniques that can predict toxicity without animal testing [1].

In this tutorial, we'll use DeepChem to build machine learning models that predict toxicity directly from molecular structure. We'll work with two MoleculeNet datasets:

- **Tox21**: 8,014 compounds tested across 12 nuclear receptor and stress response pathway assays
- **ToxCast**: 8,597 compounds tested across 617 high-throughput assays

We'll compare fingerprint-based and graph-based models, evaluate per-task performance, and see how multitask learning helps when data is sparse.

[1] Parish, S.T., et al. "An evaluation framework for new approach methodologies (NAMs) for human safety assessment." *Regulatory Toxicology and Pharmacology* 112 (2020): 104592.

## Colab

This tutorial and the rest in this sequence can be done in Google colab. If you'd like to open this notebook in colab, you can use the following link.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepchem/deepchem/blob/master/examples/tutorials/Computational_Toxicity_Prediction.ipynb)

## Setup

To run DeepChem within Colab, you'll need to run the following installation commands. You can of course run this tutorial locally if you prefer. In that case, don't run these cells since they will download and install DeepChem again on your local machine.

In [ ]:
!pip install --pre deepchem

In [ ]:
import deepchem as dc
import numpy as np
dc.__version__

## The Tox21 Dataset

The Toxicology in the 21st Century (Tox21) initiative is a collaboration between the NIH, EPA, and FDA that screens compounds against panels of biological targets related to toxicity. The Tox21 dataset in MoleculeNet contains 8,014 compounds measured on 12 assay endpoints:

- **Nuclear Receptor (NR) assays**: AR, AR-LBD, AhR, Aromatase, ER, ER-LBD, PPAR-gamma
- **Stress Response (SR) assays**: ARE, ATAD5, HSE, MMP, p53

Each endpoint is a binary classification task (active/inactive). A compound that is "active" in an assay is interacting with that biological pathway in a way that could indicate toxicity.

All of this data was generated entirely from automated in-vitro cell-based and biochemical assays -- no animal testing required. Let's load it.

In [ ]:
tox21_tasks, tox21_datasets, tox21_transformers = dc.molnet.load_tox21(
    featurizer='ECFP', splitter='scaffold'
)
train_dataset, valid_dataset, test_dataset = tox21_datasets

print(f"Tasks: {len(tox21_tasks)}")
print(f"Training compounds: {len(train_dataset)}")
print(f"Validation compounds: {len(valid_dataset)}")
print(f"Test compounds: {len(test_dataset)}")
print(f"\nAssay endpoints:")
for task in tox21_tasks:
    print(f"  {task}")

We used Extended-Connectivity Fingerprints (ECFP) as the featurizer. ECFPs encode the presence of molecular substructures as a fixed-length binary vector -- a compact representation of what chemical "fragments" appear in each molecule.

We also used scaffold splitting, which ensures train/test molecules come from different chemical scaffolds. This is a harder but more realistic evaluation than random splitting, because in practice you want to predict toxicity for novel chemical structures.

## Training a Multitask Model

A key advantage of datasets like Tox21 is that they measure many endpoints simultaneously. A **multitask model** predicts all 12 endpoints from a single shared representation. This is valuable because:

1. Tasks that share biological mechanisms can learn from each other
2. Sparse labels in one task get supplemented by data from other tasks
3. The shared representation learns more general chemical features

Let's train a `MultitaskClassifier` -- a stack of fully connected layers with shared hidden representations and per-task output heads.

In [ ]:
n_tasks = len(tox21_tasks)
n_features = train_dataset.get_data_shape()[0]

model = dc.models.MultitaskClassifier(
    n_tasks,
    n_features,
    layer_sizes=[1024, 512],
    dropouts=0.2,
    learning_rate=0.001,
    batch_size=128,
)
model.fit(train_dataset, nb_epoch=15)

## Evaluating Per-Task Performance

Since each task has very different class balance (some targets have very few active compounds), we use **ROC-AUC** as the evaluation metric. An AUC of 0.5 means random guessing, while 1.0 is perfect prediction.

In [ ]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)

print("Training set performance:")
train_scores = model.evaluate(train_dataset, [metric], tox21_transformers)

print("\nTest set performance:")
test_scores = model.evaluate(test_dataset, [metric], tox21_transformers)

## Graph Convolutional Model

Fingerprint-based models treat molecules as fixed-length vectors. An alternative approach is to work with the molecular graph directly, where atoms are nodes and bonds are edges. **Graph convolutional models** learn features directly from this graph structure, which can capture spatial and topological patterns that fingerprints miss.

Let's reload Tox21 with the `GraphConv` featurizer and train a `GraphConvModel`.

In [ ]:
gc_tasks, gc_datasets, gc_transformers = dc.molnet.load_tox21(
    featurizer='GraphConv', splitter='scaffold'
)
gc_train, gc_valid, gc_test = gc_datasets

gc_model = dc.models.GraphConvModel(
    n_tasks=len(gc_tasks),
    mode='classification',
    batch_size=128,
    learning_rate=0.001,
    dropout=0.2,
)
gc_model.fit(gc_train, nb_epoch=15)

In [ ]:
print("GraphConvModel - Training set:")
gc_train_scores = gc_model.evaluate(gc_train, [metric], gc_transformers)

print("\nGraphConvModel - Test set:")
gc_test_scores = gc_model.evaluate(gc_test, [metric], gc_transformers)

## Scaling Up: ToxCast

While Tox21 has 12 endpoints, the EPA's ToxCast program measured compounds across **617 assays** spanning a much broader range of biological targets. This makes it one of the largest public toxicity datasets available.

The sheer number of tasks makes multitask learning particularly valuable here -- many individual assays have very sparse data, but the shared representation across hundreds of tasks helps compensate.

In [ ]:
tc_tasks, tc_datasets, tc_transformers = dc.molnet.load_toxcast(
    featurizer='ECFP', splitter='scaffold'
)
tc_train, tc_valid, tc_test = tc_datasets

print(f"ToxCast tasks: {len(tc_tasks)}")
print(f"Training compounds: {len(tc_train)}")
print(f"Test compounds: {len(tc_test)}")

In [ ]:
tc_model = dc.models.MultitaskClassifier(
    len(tc_tasks),
    tc_train.get_data_shape()[0],
    layer_sizes=[1024, 512],
    dropouts=0.2,
    learning_rate=0.001,
    batch_size=128,
)
tc_model.fit(tc_train, nb_epoch=10)

print("ToxCast - Test set:")
tc_test_scores = tc_model.evaluate(tc_test, [metric], tc_transformers)

## From In-Vitro Data to Clinical Toxicity: ClinTox

Can models trained on in-vitro assay data generalize to predicting clinical toxicity outcomes? The **ClinTox** dataset contains 1,491 compounds with two labels: FDA approval status and clinical trial toxicity. Let's train a model on ClinTox and compare with the earlier Tox21 results.

ClinTox bridges the gap between high-throughput in-vitro screening (like Tox21) and real-world drug safety outcomes, demonstrating how computational approaches can inform decisions that previously required animal studies.

In [ ]:
ct_tasks, ct_datasets, ct_transformers = dc.molnet.load_clintox(
    featurizer='ECFP', splitter='scaffold'
)
ct_train, ct_valid, ct_test = ct_datasets

print(f"ClinTox tasks: {ct_tasks}")
print(f"Training compounds: {len(ct_train)}")
print(f"Test compounds: {len(ct_test)}")

ct_model = dc.models.MultitaskClassifier(
    len(ct_tasks),
    ct_train.get_data_shape()[0],
    layer_sizes=[512, 256],
    dropouts=0.2,
    learning_rate=0.001,
    batch_size=64,
)
ct_model.fit(ct_train, nb_epoch=20)

print("\nClinTox - Test set:")
ct_test_scores = ct_model.evaluate(ct_test, [metric], ct_transformers)

## The Bigger Picture

The models we built in this tutorial are examples of **Quantitative Structure-Activity Relationship (QSAR)** models -- they predict biological activity from molecular structure. QSAR models are one of several computational approaches that collectively form New Approach Methodologies:

- **QSAR/QSPR models**: Predict activity or properties from chemical structure (what we did here)
- **Read-across**: Predict toxicity of untested chemicals based on similar compounds with known toxicity
- **PBPK models**: Physiologically-based pharmacokinetic models that simulate how chemicals move through the body
- **In-vitro assays**: Cell-based experiments like those in Tox21 and ToxCast (the data source for our models)
- **Organ-on-chip**: Microfluidic devices that mimic organ function for toxicity testing

These approaches are increasingly recognized by regulatory agencies. The EPA committed in 2019 to [reducing animal testing requests](https://www.epa.gov/research/epa-new-approach-methods-work-plan-reducing-use-vertebrate-animals-chemical-testing) by 2035, with NAMs playing a central role. The 2023 FDA Modernization Act 2.0 eliminated the requirement for animal testing in drug approval, opening the door for computational and in-vitro alternatives.

Datasets like Tox21 and ToxCast -- and the computational models built from them -- are building blocks for a future where toxicity assessment relies on data and algorithms rather than animal experiments.

## Further Reading

- Wu, Z., et al. "MoleculeNet: a benchmark for molecular machine learning." *Chemical Science* 9.2 (2018): 513-530.
- Mayr, A., et al. "DeepTox: Toxicity Prediction using Deep Learning." *Frontiers in Environmental Science* 3 (2016): 80.
- Thomas, R.S., et al. "The Next Generation Blueprint of Computational Toxicology at the U.S. Environmental Protection Agency." *Toxicological Sciences* 169.2 (2019): 317-332.

# Congratulations! Time to join the Community!

Congratulations on completing this tutorial notebook! If you enjoyed working through the tutorial, and want to continue working with DeepChem, we encourage you to finish the rest of the tutorials in this series. You can also help the DeepChem community in the following ways:

## Star DeepChem on [GitHub](https://github.com/deepchem/deepchem)
This helps build awareness of the DeepChem project and the tools for open source drug discovery that we're trying to build.

## Join the DeepChem Discord
The DeepChem [Discord](https://discord.gg/cGzwCdrUqS) hosts a number of scientists, developers, and enthusiasts interested in deep learning for the life sciences. Join the conversation!